In [1]:
import os
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import gc
import json
import math
import random
import re
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

try:
    from peft import PeftModel
    HAS_PEFT = True
except Exception:
    HAS_PEFT = False

# ============================================================
# CONFIG
# ============================================================

BASE_MODEL = "microsoft/phi-3-mini-4k-instruct"
SFT_PATH = "/kaggle/input/datasets/adithyaled24b039/phi3-sft-folderr/phi3_sft_lora"

OUT_DIR = "/kaggle/working/phi3_consistency_head_eap"
PLOTS_DIR = os.path.join(OUT_DIR, "plots")
REPORTS_DIR = os.path.join(OUT_DIR, "reports")
CSV_DIR = os.path.join(OUT_DIR, "csv")
CACHE_DIR = os.path.join(OUT_DIR, "cache")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32

SEED = 42
EVAL_SEED = 42

# Keep this moderate for Kaggle T4.
N_SAMPLES = 40
MAX_NEW_TOKENS = 32

# Consistency-head layers described in the document.
CONSISTENCY_LAYERS = list(range(14, 19))

# Attention / patching controls
PREFIX_WINDOW = 8            # first N prompt tokens for attention mass
PATCH_WINDOW = 12            # patch the first N tokens from clean into corrupt
TOP_EAP_POSITIONS = 12       # saved per-sample top token positions per layer

# Prompt forms
USE_CHAT_TEMPLATE = False     # keep False for maximum control over delimiter experiments
USE_SFT_ADAPTER = True

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)
os.makedirs(CSV_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

TOKENIZER = None
MODEL = None
BLOCKS = None
FINAL_NORM = None
LM_HEAD = None
MODEL_ARCH = None

# ============================================================
# PROMPTS
# ============================================================

ANCHOR_Q = "## Question:"
ANCHOR_TRACE = "## Worked solution:"
ANCHOR_ANSWER = "## "

PLAIN_Q = "Question:"
PLAIN_TRACE = "Worked solution:"
PLAIN_ANSWER = ""

# ============================================================
# UTILS
# ============================================================

def ensure_dir(path):
    os.makedirs(path, exist_ok=True)

def free_memory(*objs):
    for o in objs:
        try:
            del o
        except Exception:
            pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.synchronize()
        except Exception:
            pass

def canonical_number_str(x):
    if x is None:
        return None
    try:
        v = float(re.sub(r"[$,]", "", str(x)))
    except Exception:
        return None
    if abs(v - round(v)) < 1e-6:
        return str(int(round(v)))
    return f"{v:.6f}".rstrip("0").rstrip(".")

def same_numeric(a, b, tol=1e-6):
    try:
        na = float(re.sub(r"[$,]", "", str(a)))
        nb = float(re.sub(r"[$,]", "", str(b)))
        return abs(na - nb) <= tol
    except Exception:
        return False

def extract_number(text):
    m = re.findall(r"<answer>(.*?)</answer>", text, flags=re.IGNORECASE | re.DOTALL)
    span = m[-1] if m else text
    span = re.sub(r"[$,]", "", span)
    nums = re.findall(r"-?\d+\.?\d*", span)
    if nums:
        return nums[-1]
    nums = re.findall(r"-?\d+\.?\d*", re.sub(r"[$,]", "", text))
    return nums[-1] if nums else None

def parse_prediction(text):
    return canonical_number_str(extract_number(text))

def preview_text(s, max_len=180):
    s = str(s).replace("\n", " ")
    return s[:max_len] + ("..." if len(s) > max_len else "")

def to_jsonable(x):
    try:
        return json.dumps(x, ensure_ascii=False)
    except Exception:
        return json.dumps(str(x), ensure_ascii=False)

def save_df(df, path):
    ensure_dir(os.path.dirname(path))
    df.to_csv(path, index=False)

def save_json(obj, path):
    ensure_dir(os.path.dirname(path))
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)

def load_cached_indices(name, ds_len, n, seed=EVAL_SEED):
    ensure_dir(CACHE_DIR)
    n = min(n, ds_len)
    path = os.path.join(CACHE_DIR, f"{name}_n{n}_seed{seed}.json")
    if os.path.exists(path):
        with open(path, "r") as f:
            return json.load(f)
    rng = np.random.RandomState(seed)
    idx = rng.choice(ds_len, size=n, replace=False).tolist()
    with open(path, "w") as f:
        json.dump(idx, f)
    return idx

def sample_dataset(ds, name, n, seed=EVAL_SEED):
    idx = load_cached_indices(name, len(ds), n, seed)
    return ds.select(idx), idx

def patch_window_positions(prompt_text, tokenizer, window):
    try:
        enc = tokenizer(prompt_text, return_offsets_mapping=True, add_special_tokens=True)
        offsets = enc["offset_mapping"]
        positions = []
        for i, (a, b) in enumerate(offsets):
            if a is None or b is None:
                continue
            if a == b == 0:
                continue
            if i < window:
                positions.append(i)
        return positions
    except Exception:
        ids = tokenizer(prompt_text, add_special_tokens=True).input_ids
        return list(range(min(window, len(ids))))

def span_token_positions(prompt_text, tokenizer, spans):
    try:
        enc = tokenizer(prompt_text, return_offsets_mapping=True, add_special_tokens=True)
        offsets = enc["offset_mapping"]
        hits = set()
        char_spans = []
        for sp in spans:
            start = prompt_text.find(sp)
            if start >= 0:
                char_spans.append((start, start + len(sp)))
        for i, (a, b) in enumerate(offsets):
            if a is None or b is None:
                continue
            for s, e in char_spans:
                if not (b <= s or a >= e):
                    hits.add(i)
        return sorted(hits)
    except Exception:
        return []

# ============================================================
# ADVANCED PLOTTING FUNCTIONS
# ============================================================

def line_plot(x, series, labels, title, path, xlabel="Layer", ylabel="Value"):
    plt.figure(figsize=(10, 5))
    for y, label in zip(series, labels):
        plt.plot(x, y, marker="o", linewidth=1.5, label=label)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.legend()
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()

def bar_plot(labels, values, title, path, ylabel="Value"):
    plt.figure(figsize=(9, 4))
    plt.bar(labels, values, color=['#4C72B0', '#DD8452', '#55A868', '#C44E52'][:len(labels)])
    plt.title(title)
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()

def heatmap(mat, xlabels, ylabels, title, path, xlabel="", ylabel="", cmap="viridis"):
    plt.figure(figsize=(max(8, len(xlabels) * 0.55), max(5, len(ylabels) * 0.35)))
    im = plt.imshow(mat, aspect="auto", cmap=cmap)
    plt.colorbar(im)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.xticks(range(len(xlabels)), xlabels, rotation=45, ha="right")
    plt.yticks(range(len(ylabels)), ylabels)
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()

def surface_3d_plot(mat, xlabels, ylabels, title, path, xlabel="Head", ylabel="Layer", zlabel="Delta"):
    fig = plt.figure(figsize=(10, 7))
    ax = fig.add_subplot(111, projection='3d')
    X, Y = np.meshgrid(range(len(xlabels)), range(len(ylabels)))
    surf = ax.plot_surface(X, Y, mat, cmap='coolwarm', edgecolor='none', alpha=0.9)
    fig.colorbar(surf, ax=ax, shrink=0.5, aspect=5, pad=0.1)
    ax.set_xticks(range(len(xlabels)))
    ax.set_xticklabels(xlabels)
    ax.set_yticks(range(len(ylabels)))
    ax.set_yticklabels(ylabels)
    ax.set_title(title, pad=20)
    ax.set_xlabel(xlabel, labelpad=10)
    ax.set_ylabel(ylabel, labelpad=10)
    ax.set_zlabel(zlabel, labelpad=10)
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()

def violin_layer_plot(df, val_col, layer_col, title, path, ylabel="Value"):
    layers = sorted(df[layer_col].unique())
    data = [df[df[layer_col] == l][val_col].values for l in layers]
    data = [d[~np.isnan(d)] for d in data]

    plt.figure(figsize=(10, 5))
    if any(len(d) > 0 for d in data):
        parts = plt.violinplot(data, positions=layers, showmeans=True, showextrema=True)
        for pc in parts['bodies']:
            pc.set_facecolor('#55A868')
            pc.set_edgecolor('black')
            pc.set_alpha(0.6)
        plt.title(title)
        plt.xlabel("Layer")
        plt.ylabel(ylabel)
        plt.xticks(layers)
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()

def correlation_heatmap(df, cols, title, path):
    sub_df = df[cols].dropna()
    if len(sub_df) < 2: return
    corr = sub_df.corr().values
    plt.figure(figsize=(8, 6))
    im = plt.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
    plt.colorbar(im)
    plt.xticks(range(len(cols)), cols, rotation=30, ha='right')
    plt.yticks(range(len(cols)), cols)
    for i in range(len(cols)):
        for j in range(len(cols)):
            color = 'black' if abs(corr[i, j]) < 0.5 else 'white'
            plt.text(j, i, f"{corr[i, j]:.2f}", ha='center', va='center', color=color)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()

# ============================================================
# DATA
# ============================================================

def build_trace(question, rationale, clean=True):
    q = ANCHOR_Q if clean else PLAIN_Q
    t = ANCHOR_TRACE if clean else PLAIN_TRACE
    a = ANCHOR_ANSWER if clean else PLAIN_ANSWER
    return (
        f"{q} {question}\n"
        f"{t}\n"
        f"{rationale.strip()}\n"
        f"{a}\n"
    )

def inject_arithmetic_error(rationale):
    text = rationale.strip()
    eq_pat = re.compile(r"(-?\d+(?:\.\d+)?)\s*([+\-*/])\s*(-?\d+(?:\.\d+)?)\s*=\s*(-?\d+(?:\.\d+)?)")
    m = eq_pat.search(text)
    if m:
        a, op, b, c = m.groups()
        try:
            a_f, b_f, c_f = float(a), float(b), float(c)
            if op == "+":
                wrong = c_f + 1
            elif op == "-":
                wrong = c_f - 1
            elif op == "*":
                wrong = c_f + max(1.0, abs(c_f) * 0.1)
            else:
                wrong = c_f + 1 if c_f >= 0 else c_f - 1
            wrong_str = canonical_number_str(wrong)
            return text[:m.start(4)] + wrong_str + text[m.end(4):]
        except Exception:
            pass

    num_pat = re.compile(r"-?\d+(?:\.\d+)?")
    m = num_pat.search(text)
    if m:
        num = m.group(0)
        try:
            n = float(num)
            wrong = n + 1 if n >= 0 else n - 1
            wrong_str = canonical_number_str(wrong)
            return text[:m.start()] + wrong_str + text[m.end():]
        except Exception:
            pass

    return text + "\n(incorrect step injected)"

def load_gsm8k_samples(n=N_SAMPLES):
    ds = load_dataset("gsm8k", "main", split="test")
    ds, idx = sample_dataset(ds, "gsm8k_consistency", n, seed=EVAL_SEED)
    samples = []
    for i, s in enumerate(ds):
        answer = s["answer"]
        rationale, gold = answer.split("####", 1)
        gold = canonical_number_str(gold.strip())
        samples.append({
            "sample_id": i,
            "source_index": idx[i],
            "question": s["question"],
            "rationale": rationale.strip(),
            "gold": gold,
        })
    return samples

# ============================================================
# MODEL LOADING
# ============================================================

def load_tokenizer():
    global TOKENIZER
    if TOKENIZER is None:
        src = SFT_PATH if (USE_SFT_ADAPTER and os.path.exists(SFT_PATH)) else BASE_MODEL
        TOKENIZER = AutoTokenizer.from_pretrained(src)
        if TOKENIZER.pad_token is None:
            TOKENIZER.pad_token = TOKENIZER.eos_token
    return TOKENIZER

def load_model():
    global MODEL, BLOCKS, FINAL_NORM, LM_HEAD, MODEL_ARCH
    tok = load_tokenizer()

    kwargs = {"torch_dtype": DTYPE}
    try:
        kwargs["attn_implementation"] = "eager"
        model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **kwargs)
    except Exception:
        kwargs.pop("attn_implementation", None)
        model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **kwargs)

    model = model.to(DEVICE)

    if USE_SFT_ADAPTER and os.path.exists(SFT_PATH) and HAS_PEFT:
        model = PeftModel.from_pretrained(model, SFT_PATH).merge_and_unload().to(DEVICE)

    model.eval()
    try:
        model.config.use_cache = False
    except Exception:
        pass

    arch = None
    blocks = None
    candidates = [
        ("model.layers", getattr(getattr(model, "model", None), "layers", None)),
        ("model.decoder.layers", getattr(getattr(getattr(model, "model", None), "decoder", None), "layers", None)),
        ("transformer.h", getattr(getattr(model, "transformer", None), "h", None)),
        ("gpt_neox.layers", getattr(getattr(model, "gpt_neox", None), "layers", None)),
        ("decoder.layers", getattr(getattr(model, "decoder", None), "layers", None)),
    ]
    for name, maybe_blocks in candidates:
        if maybe_blocks is not None:
            maybe_blocks = list(maybe_blocks)
            if len(maybe_blocks) > 0:
                blocks = maybe_blocks
                arch = name
                break
    if blocks is None:
        raise RuntimeError("Could not locate transformer blocks.")

    final_norm = None
    norm_candidates = [
        ("model.norm", getattr(getattr(model, "model", None), "norm", None)),
        ("transformer.ln_f", getattr(getattr(model, "transformer", None), "ln_f", None)),
        ("gpt_neox.final_layer_norm", getattr(getattr(model, "gpt_neox", None), "final_layer_norm", None)),
        ("decoder.final_layer_norm", getattr(getattr(model, "decoder", None), "final_layer_norm", None)),
    ]
    for name, maybe_norm in norm_candidates:
        if maybe_norm is not None:
            final_norm = maybe_norm
            break

    if hasattr(model, "lm_head"):
        lm_head = model.lm_head
    elif hasattr(model, "embed_out"):
        lm_head = model.embed_out
    else:
        raise RuntimeError("Could not locate lm_head / embed_out.")

    MODEL = model
    BLOCKS = blocks
    FINAL_NORM = final_norm
    LM_HEAD = lm_head
    MODEL_ARCH = arch

    print(f"Loaded model with {len(BLOCKS)} blocks ({MODEL_ARCH}).")
    return model, tok

# ============================================================
# GENERATION / PROBABILITY
# ============================================================

def greedy_generate(prompt, max_new_tokens=32):
    inputs = TOKENIZER(prompt, return_tensors="pt").to(DEVICE)
    with torch.inference_mode():
        out = MODEL.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=TOKENIZER.eos_token_id,
            eos_token_id=TOKENIZER.eos_token_id,
        )
    return TOKENIZER.decode(out[0], skip_special_tokens=True)

def completion_logprob(prompt, completion, max_length=4096):
    full = prompt + completion
    prompt_ids = TOKENIZER(prompt, return_tensors="pt", truncation=True, max_length=max_length).input_ids.to(DEVICE)
    full_ids = TOKENIZER(full, return_tensors="pt", truncation=True, max_length=max_length).input_ids.to(DEVICE)

    with torch.inference_mode():
        logits = MODEL(full_ids).logits[:, :-1, :]
        logp_all = torch.log_softmax(logits.float(), dim=-1)
        tgt = full_ids[:, 1:]
        tok_lp = logp_all.gather(-1, tgt.unsqueeze(-1)).squeeze(-1)

    start = max(0, prompt_ids.shape[1] - 1)
    comp_lp = tok_lp[:, start:]
    if comp_lp.numel() == 0:
        return {"sum_logprob": float("-inf"), "mean_logprob": float("-inf"), "n_tokens": 0}
    return {
        "sum_logprob": float(comp_lp.sum().item()),
        "mean_logprob": float(comp_lp.mean().item()),
        "n_tokens": int(comp_lp.numel()),
    }

def final_answer_token_id(gold):
    ids = TOKENIZER.encode(" " + str(gold), add_special_tokens=False)
    if len(ids) == 0:
        ids = TOKENIZER.encode(str(gold), add_special_tokens=False)
    if len(ids) == 0:
        return int(TOKENIZER.eos_token_id)
    return int(ids[0])

def final_token_logit(prompt, target_id):
    inputs = TOKENIZER(prompt, return_tensors="pt").to(DEVICE)
    with torch.inference_mode():
        logits = MODEL(**inputs).logits[0, -1].float()
    return float(logits[target_id].item())

def hidden_dla_curve(prompt, target_id):
    inputs = TOKENIZER(prompt, return_tensors="pt").to(DEVICE)
    with torch.inference_mode():
        out = MODEL(
            **inputs,
            output_hidden_states=True,
            output_attentions=True,
            use_cache=False,
            return_dict=True,
        )
    qpos = inputs["input_ids"].shape[1] - 1
    curve = []
    for h in out.hidden_states:
        vec = h[:, qpos, :]
        if FINAL_NORM is not None:
            try:
                vec = FINAL_NORM(vec)
            except Exception:
                pass
        if hasattr(LM_HEAD, "weight"):
            target_vec = LM_HEAD.weight[target_id].float()
            val = torch.dot(vec.squeeze(0).float(), target_vec).item()
        else:
            val = float("nan")
        curve.append(val)
    return curve, out, inputs

# ============================================================
# EAP / ATTN / PATCHING
# ============================================================

def make_capture_hook(layer_idx, storage):
    def hook(module, inp, out):
        hidden = out[0] if isinstance(out, tuple) else out
        if torch.is_grad_enabled() and getattr(hidden, "requires_grad", False):
            hidden.retain_grad()
        storage[layer_idx] = hidden
        return out
    return hook

def eap_proxy_for_layers(prompt, target_id, layer_ids):
    inputs = TOKENIZER(prompt, return_tensors="pt").to(DEVICE)
    storage = {}
    hooks = []

    embeds = MODEL.get_input_embeddings()
    orig_req_grad = embeds.weight.requires_grad
    embeds.weight.requires_grad_(True)

    try:
        for li in layer_ids:
            hooks.append(BLOCKS[li].register_forward_hook(make_capture_hook(li, storage)))

        MODEL.zero_grad(set_to_none=True)
        with torch.enable_grad():
            out = MODEL(**inputs, use_cache=False, return_dict=True)
            logit = out.logits[0, -1, target_id]
            logit.backward()

        rows = []
        for li in layer_ids:
            h = storage.get(li, None)
            if h is None or getattr(h, "grad", None) is None:
                continue
            token_attr = (h.detach() * h.grad.detach()).sum(dim=-1)[0]
            rows.append({
                "layer": li,
                "eap_sum": float(token_attr.sum().item()),
                "eap_mean": float(token_attr.mean().item()),
                "eap_max": float(token_attr.max().item()),
                "eap_min": float(token_attr.min().item()),
                "token_attr": token_attr.detach().float().cpu().tolist(),
            })
        scalar = float(logit.detach().item())
        return scalar, rows
    finally:
        for h in hooks:
            h.remove()
        embeds.weight.requires_grad_(orig_req_grad)
        MODEL.zero_grad(set_to_none=True)

def patch_block_output(clean_hidden, layer_idx, patch_positions):
    def hook(module, inp, out):
        if isinstance(out, tuple):
            hidden = out[0]
            patched = hidden.clone()
            for p in patch_positions:
                if p < patched.shape[1] and p < clean_hidden.shape[1]:
                    patched[:, p, :] = clean_hidden[:, p, :].to(device=patched.device, dtype=patched.dtype)
            return (patched,) + out[1:]
        hidden = out
        patched = hidden.clone()
        for p in patch_positions:
            if p < patched.shape[1] and p < clean_hidden.shape[1]:
                patched[:, p, :] = clean_hidden[:, p, :].to(device=patched.device, dtype=patched.dtype)
        return patched
    return hook

def patched_final_logit(corrupt_prompt, target_id, clean_hidden_state_after_layer, layer_idx, patch_positions):
    handle = BLOCKS[layer_idx].register_forward_hook(
        patch_block_output(clean_hidden_state_after_layer, layer_idx, patch_positions)
    )
    try:
        return final_token_logit(corrupt_prompt, target_id)
    finally:
        handle.remove()

def attention_mass(attentions, query_pos, prefix_positions):
    if not attentions:
        return np.array([]), np.array([])
    
    layer_mean = []
    layer_head = []
    for layer_attn in attentions:
        a = layer_attn[0]
        if query_pos >= a.shape[-2]:
            layer_mean.append(float("nan"))
            layer_head.append(np.full((a.shape[0],), np.nan))
            continue
        valid_prefix = [p for p in prefix_positions if p < a.shape[-1]]
        if not valid_prefix:
            layer_mean.append(float("nan"))
            layer_head.append(np.full((a.shape[0],), np.nan))
            continue
        mass_per_head = a[:, query_pos, valid_prefix].sum(dim=-1)
        layer_mean.append(float(mass_per_head.mean().item()))
        layer_head.append(mass_per_head.detach().float().cpu().numpy())
    return np.array(layer_mean), np.stack(layer_head, axis=0)

# ============================================================
# SAMPLE ANALYSIS
# ============================================================

def run_sample(sample):
    question = sample["question"]
    rationale = sample["rationale"]
    gold = sample["gold"]
    target_id = final_answer_token_id(gold)

    prompts = {
        "anchored_clean": build_trace(question, rationale, clean=True),
        "anchored_corrupt": build_trace(question, inject_arithmetic_error(rationale), clean=True),
        "plain_clean": build_trace(question, rationale, clean=False),
        "plain_corrupt": build_trace(question, inject_arithmetic_error(rationale), clean=False),
    }

    completion = f"{gold}"
    completion_tagged = f"<answer>{gold}</answer>"

    rows = []
    eap_rows = []
    patch_rows = []
    attn_rows = []
    dla_rows = []
    token_rows = []

    for mode in ["anchored", "plain"]:
        clean_prompt = prompts[f"{mode}_clean"]
        corrupt_prompt = prompts[f"{mode}_corrupt"]

        clean_gen = greedy_generate(clean_prompt, MAX_NEW_TOKENS)
        corrupt_gen = greedy_generate(corrupt_prompt, MAX_NEW_TOKENS)

        clean_pred = parse_prediction(clean_gen)
        corrupt_pred = parse_prediction(corrupt_gen)

        clean_ok = same_numeric(clean_pred, gold)
        corrupt_ok = same_numeric(corrupt_pred, gold)

        clean_lp = completion_logprob(clean_prompt, completion_tagged)
        corrupt_lp = completion_logprob(corrupt_prompt, completion_tagged)

        clean_target_logit = final_token_logit(clean_prompt, target_id)
        corrupt_target_logit = final_token_logit(corrupt_prompt, target_id)

        clean_dla, clean_out, clean_inputs = hidden_dla_curve(clean_prompt, target_id)
        corrupt_dla, corrupt_out, corrupt_inputs = hidden_dla_curve(corrupt_prompt, target_id)

        clean_qpos = clean_inputs["input_ids"].shape[1] - 1
        corrupt_qpos = corrupt_inputs["input_ids"].shape[1] - 1

        clean_prefix_positions = list(range(min(PREFIX_WINDOW, clean_inputs["input_ids"].shape[1])))
        corrupt_prefix_positions = list(range(min(PREFIX_WINDOW, corrupt_inputs["input_ids"].shape[1])))
        
        c_attns = clean_out.attentions or []
        k_attns = corrupt_out.attentions or []
        
        clean_prefix_attn, clean_prefix_head = attention_mass(c_attns, clean_qpos, clean_prefix_positions)
        corrupt_prefix_attn, corrupt_prefix_head = attention_mass(k_attns, corrupt_qpos, corrupt_prefix_positions)

        marker_spans = [ANCHOR_Q if mode == "anchored" else PLAIN_Q,
                        ANCHOR_TRACE if mode == "anchored" else PLAIN_TRACE,
                        ANCHOR_ANSWER if mode == "anchored" else PLAIN_ANSWER]
        clean_marker_pos = span_token_positions(clean_prompt, TOKENIZER, marker_spans)
        corrupt_marker_pos = span_token_positions(corrupt_prompt, TOKENIZER, marker_spans)
        clean_num_pos = span_token_positions(clean_prompt, TOKENIZER, [str(gold)])
        corrupt_num_pos = span_token_positions(corrupt_prompt, TOKENIZER, [str(gold)])

        clean_eap_scalar, clean_eap_layer_rows = eap_proxy_for_layers(clean_prompt, target_id, CONSISTENCY_LAYERS)
        corrupt_eap_scalar, corrupt_eap_layer_rows = eap_proxy_for_layers(corrupt_prompt, target_id, CONSISTENCY_LAYERS)

        patch_positions = patch_window_positions(clean_prompt, TOKENIZER, PATCH_WINDOW)
        for layer_idx in CONSISTENCY_LAYERS:
            clean_after = clean_out.hidden_states[layer_idx + 1]
            patched_logit = patched_final_logit(corrupt_prompt, target_id, clean_after, layer_idx, patch_positions)
            patch_rows.append({
                "sample_id": sample["sample_id"],
                "mode": mode,
                "layer": layer_idx,
                "clean_target_logit": clean_target_logit,
                "corrupt_target_logit": corrupt_target_logit,
                "patched_target_logit": patched_logit,
                "recovery": (patched_logit - corrupt_target_logit) / (clean_target_logit - corrupt_target_logit + 1e-8),
                "patch_window": len(patch_positions),
            })

        for li, (c, k) in enumerate(zip(clean_dla, corrupt_dla)):
            dla_rows.append({
                "sample_id": sample["sample_id"],
                "mode": mode,
                "layer": li,
                "clean_dla": c,
                "corrupt_dla": k,
                "delta_dla": c - k,
            })

        num_layers = len(c_attns)
        for local_i, layer_idx in enumerate(range(num_layers)):
            attn_rows.append({
                "sample_id": sample["sample_id"],
                "mode": mode,
                "layer": layer_idx,
                "clean_prefix_attn": float(clean_prefix_attn[layer_idx]),
                "corrupt_prefix_attn": float(corrupt_prefix_attn[layer_idx]),
                "delta_prefix_attn": float(clean_prefix_attn[layer_idx] - corrupt_prefix_attn[layer_idx]),
                "is_consistency_layer": layer_idx in CONSISTENCY_LAYERS,
            })

        for i, layer_idx in enumerate(range(num_layers)):
            if layer_idx not in CONSISTENCY_LAYERS:
                continue
            ch = clean_prefix_head[layer_idx]
            kh = corrupt_prefix_head[layer_idx]
            for head_idx in range(len(ch)):
                attn_rows.append({
                    "sample_id": sample["sample_id"],
                    "mode": mode,
                    "layer": layer_idx,
                    "head": head_idx,
                    "clean_prefix_attn_head": float(ch[head_idx]),
                    "corrupt_prefix_attn_head": float(kh[head_idx]),
                    "delta_prefix_attn_head": float(ch[head_idx] - kh[head_idx]),
                    "is_head_row": True,
                    "is_consistency_layer": True,
                })

        token_rows.append({
            "sample_id": sample["sample_id"],
            "mode": mode,
            "clean_prompt_tokens": len(TOKENIZER(clean_prompt).input_ids),
            "corrupt_prompt_tokens": len(TOKENIZER(corrupt_prompt).input_ids),
            "clean_marker_positions": to_jsonable(clean_marker_pos),
            "corrupt_marker_positions": to_jsonable(corrupt_marker_pos),
            "clean_num_positions": to_jsonable(clean_num_pos),
            "corrupt_num_positions": to_jsonable(corrupt_num_pos),
        })

        rows.append({
            "sample_id": sample["sample_id"],
            "mode": mode,
            "question": question,
            "gold": gold,
            "rationale": rationale,
            "clean_prompt": clean_prompt,
            "corrupt_prompt": corrupt_prompt,
            "clean_generated": clean_gen,
            "corrupt_generated": corrupt_gen,
            "clean_prediction": clean_pred,
            "corrupt_prediction": corrupt_pred,
            "clean_correct": clean_ok,
            "corrupt_correct": corrupt_ok,
            "clean_gold_logprob_sum": clean_lp["sum_logprob"],
            "corrupt_gold_logprob_sum": corrupt_lp["sum_logprob"],
            "clean_gold_logprob_mean": clean_lp["mean_logprob"],
            "corrupt_gold_logprob_mean": corrupt_lp["mean_logprob"],
            "clean_target_logit": clean_target_logit,
            "corrupt_target_logit": corrupt_target_logit,
            "delta_gold_logprob_mean": clean_lp["mean_logprob"] - corrupt_lp["mean_logprob"],
            "delta_target_logit": clean_target_logit - corrupt_target_logit,
            "clean_prompt_len": len(TOKENIZER(clean_prompt).input_ids),
            "corrupt_prompt_len": len(TOKENIZER(corrupt_prompt).input_ids),
        })

        for rr in clean_eap_layer_rows:
            eap_rows.append({
                "sample_id": sample["sample_id"],
                "mode": mode,
                "condition": "clean",
                **rr,
            })
        for rr in corrupt_eap_layer_rows:
            eap_rows.append({
                "sample_id": sample["sample_id"],
                "mode": mode,
                "condition": "corrupt",
                **rr,
            })

    return rows, patch_rows, attn_rows, eap_rows, token_rows, dla_rows

# ============================================================
# PLOTS / REPORTS
# ============================================================

def aggregate_and_save(samples_df, patch_df, attn_df, eap_df, token_df, dla_df):
    summary = []
    for mode in ["anchored", "plain"]:
        df = samples_df[samples_df["mode"] == mode]
        if df.empty:
            continue
        summary.append({
            "mode": mode,
            "n": int(len(df)),
            "clean_accuracy": float(df["clean_correct"].mean()),
            "corrupt_accuracy": float(df["corrupt_correct"].mean()),
            "clean_gold_logprob_mean": float(df["clean_gold_logprob_mean"].mean()),
            "corrupt_gold_logprob_mean": float(df["corrupt_gold_logprob_mean"].mean()),
            "delta_gold_logprob_mean": float(df["delta_gold_logprob_mean"].mean()),
            "delta_target_logit_mean": float(df["delta_target_logit"].mean()),
            "avg_clean_prompt_len": float(df["clean_prompt_len"].mean()),
            "avg_corrupt_prompt_len": float(df["corrupt_prompt_len"].mean()),
        })

    summary_df = pd.DataFrame(summary)
    save_df(summary_df, os.path.join(CSV_DIR, "summary.csv"))

    if not patch_df.empty:
        patch_agg = patch_df.groupby(["mode", "layer"], as_index=False).agg(
            clean_target_logit=("clean_target_logit", "mean"),
            corrupt_target_logit=("corrupt_target_logit", "mean"),
            patched_target_logit=("patched_target_logit", "mean"),
            recovery=("recovery", "mean"),
        )
    else:
        patch_agg = pd.DataFrame()
    save_df(patch_agg, os.path.join(CSV_DIR, "patch_layer_agg.csv"))

    if not eap_df.empty:
        eap_agg = eap_df.groupby(["mode", "condition", "layer"], as_index=False).agg(
            eap_sum=("eap_sum", "mean"),
            eap_mean=("eap_mean", "mean"),
            eap_max=("eap_max", "mean"),
            eap_min=("eap_min", "mean"),
        )
    else:
        eap_agg = pd.DataFrame()
    save_df(eap_agg, os.path.join(CSV_DIR, "eap_layer_agg.csv"))

    if not dla_df.empty:
        dla_agg = dla_df.groupby(["mode", "layer"], as_index=False).agg(
            clean_dla=("clean_dla", "mean"),
            corrupt_dla=("corrupt_dla", "mean"),
            delta_dla=("delta_dla", "mean"),
        )
    else:
        dla_agg = pd.DataFrame()
    save_df(dla_agg, os.path.join(CSV_DIR, "dla_layer_agg.csv"))

    if not attn_df.empty:
        attn_layer_df = attn_df[~attn_df.get("is_head_row", False).fillna(False)] if "is_head_row" in attn_df.columns else attn_df.copy()
        if not attn_layer_df.empty:
            attn_agg = attn_layer_df.groupby(["mode", "layer"], as_index=False).agg(
                clean_prefix_attn=("clean_prefix_attn", "mean"),
                corrupt_prefix_attn=("corrupt_prefix_attn", "mean"),
                delta_prefix_attn=("delta_prefix_attn", "mean"),
            )
        else:
            attn_agg = pd.DataFrame()
    else:
        attn_agg = pd.DataFrame()
    save_df(attn_agg, os.path.join(CSV_DIR, "attention_layer_agg.csv"))

    if not attn_df.empty and "head" in attn_df.columns:
        head_df = attn_df[attn_df["head"].notna()].copy()
        if not head_df.empty:
            head_agg = head_df.groupby(["mode", "layer", "head"], as_index=False).agg(
                clean_prefix_attn_head=("clean_prefix_attn_head", "mean"),
                corrupt_prefix_attn_head=("corrupt_prefix_attn_head", "mean"),
                delta_prefix_attn_head=("delta_prefix_attn_head", "mean"),
            )
            save_df(head_agg, os.path.join(CSV_DIR, "attention_head_agg.csv"))
        else:
            head_agg = pd.DataFrame()
    else:
        head_agg = pd.DataFrame()

    save_df(samples_df, os.path.join(CSV_DIR, "samples.csv"))
    save_df(patch_df, os.path.join(CSV_DIR, "patch_rows.csv"))
    save_df(attn_df, os.path.join(CSV_DIR, "attention_rows.csv"))
    save_df(eap_df, os.path.join(CSV_DIR, "eap_rows.csv"))
    save_df(token_df, os.path.join(CSV_DIR, "tokenization.csv"))
    save_df(dla_df, os.path.join(CSV_DIR, "dla_rows.csv"))

    for mode in ["anchored", "plain"]:
        if summary_df.empty or len(summary_df[summary_df["mode"] == mode]) == 0:
            continue
        sdf = summary_df[summary_df["mode"] == mode].iloc[0]
        bar_plot(
            ["clean", "corrupt"],
            [sdf["clean_accuracy"], sdf["corrupt_accuracy"]],
            f"{mode.title()} accuracy: clean vs injected error",
            os.path.join(PLOTS_DIR, f"{mode}_accuracy_bar.png"),
            ylabel="Accuracy",
        )
        bar_plot(
            ["clean", "corrupt"],
            [sdf["clean_gold_logprob_mean"], sdf["corrupt_gold_logprob_mean"]],
            f"{mode.title()} mean gold logprob: clean vs injected error",
            os.path.join(PLOTS_DIR, f"{mode}_gold_logprob_bar.png"),
            ylabel="Mean gold continuation logprob",
        )
        bar_plot(
            ["clean", "corrupt"],
            [sdf["delta_target_logit_mean"], 0],
            f"{mode.title()} mean target-logit delta (clean-corrupt)",
            os.path.join(PLOTS_DIR, f"{mode}_target_logit_delta_bar.png"),
            ylabel="Delta",
        )

        # Advanced Correlation Plot
        sdf_mode = samples_df[samples_df["mode"] == mode]
        if not sdf_mode.empty:
            correlation_heatmap(
                sdf_mode,
                ["clean_prompt_len", "clean_gold_logprob_mean", "delta_gold_logprob_mean", "delta_target_logit"],
                f"{mode.title()} Cross-Metric Correlation Map",
                os.path.join(PLOTS_DIR, f"{mode}_metric_correlations.png")
            )

    for mode in ["anchored", "plain"]:
        if not patch_agg.empty:
            p = patch_agg[patch_agg["mode"] == mode].sort_values("layer")
            if not p.empty:
                line_plot(
                    p["layer"].tolist(),
                    [p["clean_target_logit"].tolist(), p["corrupt_target_logit"].tolist(), p["patched_target_logit"].tolist()],
                    ["clean", "corrupt", "patched"],
                    f"{mode.title()} patch recovery across consistency layers",
                    os.path.join(PLOTS_DIR, f"{mode}_patch_recovery_curve.png"),
                    xlabel="Layer",
                    ylabel="Target logit",
                )
                mat = np.array(p[["recovery"]].T)
                heatmap(
                    mat,
                    [str(x) for x in p["layer"].tolist()],
                    ["recovery"],
                    f"{mode.title()} patch recovery heatmap",
                    os.path.join(PLOTS_DIR, f"{mode}_patch_recovery_heatmap.png"),
                    xlabel="Layer",
                    ylabel="Metric",
                    cmap="viridis",
                )

        if not dla_agg.empty:
            d = dla_agg[dla_agg["mode"] == mode].sort_values("layer")
            if not d.empty:
                line_plot(
                    d["layer"].tolist(),
                    [d["clean_dla"].tolist(), d["corrupt_dla"].tolist(), d["delta_dla"].tolist()],
                    ["clean", "corrupt", "delta"],
                    f"{mode.title()} DLA proxy across layers",
                    os.path.join(PLOTS_DIR, f"{mode}_dla_curve.png"),
                    xlabel="Layer",
                    ylabel="DLA proxy",
                )
            
            # Violin Distribution mapping
            vdf = dla_df[dla_df["mode"] == mode]
            if not vdf.empty:
                violin_layer_plot(
                    vdf, 
                    "delta_dla", 
                    "layer", 
                    f"{mode.title()} Layer-Wise DLA Delta Distribution", 
                    os.path.join(PLOTS_DIR, f"{mode}_dla_delta_violin.png")
                )

        if not attn_agg.empty:
            a = attn_agg[attn_agg["mode"] == mode].sort_values("layer")
            if not a.empty:
                line_plot(
                    a["layer"].tolist(),
                    [a["clean_prefix_attn"].tolist(), a["corrupt_prefix_attn"].tolist(), a["delta_prefix_attn"].tolist()],
                    ["clean", "corrupt", "delta"],
                    f"{mode.title()} prefix attention mass across layers",
                    os.path.join(PLOTS_DIR, f"{mode}_attention_curve.png"),
                    xlabel="Layer",
                    ylabel=f"Attention mass to first {PREFIX_WINDOW} tokens",
                )

        if not eap_agg.empty:
            ea = eap_agg[(eap_agg["mode"] == mode) & (eap_agg["condition"] == "clean")].sort_values("layer")
            eb = eap_agg[(eap_agg["mode"] == mode) & (eap_agg["condition"] == "corrupt")].sort_values("layer")
            if len(ea) > 0 and len(eb) > 0:
                # Crucial Fix: extract numpy values before math subtraction to avoid pandas indexing mismatch.
                ea_vals = ea["eap_sum"].values
                eb_vals = eb["eap_sum"].values
                line_plot(
                    ea["layer"].tolist(),
                    [ea_vals.tolist(), eb_vals.tolist(), (ea_vals - eb_vals).tolist()],
                    ["clean", "corrupt", "delta"],
                    f"{mode.title()} EAP proxy on consistency layers",
                    os.path.join(PLOTS_DIR, f"{mode}_eap_curve.png"),
                    xlabel="Layer",
                    ylabel="EAP proxy (sum)",
                )

    if not head_agg.empty:
        for mode in ["anchored", "plain"]:
            h = head_agg[head_agg["mode"] == mode].copy()
            if len(h) == 0:
                continue
            layers = sorted(h["layer"].unique().tolist())
            heads = sorted(h["head"].unique().tolist())
            mat = np.full((len(layers), len(heads)), np.nan)
            for _, r in h.iterrows():
                i = layers.index(int(r["layer"]))
                j = heads.index(int(r["head"]))
                mat[i, j] = r["delta_prefix_attn_head"]
            
            heatmap(
                mat,
                [str(x) for x in heads],
                [str(x) for x in layers],
                f"{mode.title()} consistency-layer head delta heatmap",
                os.path.join(PLOTS_DIR, f"{mode}_consistency_head_heatmap.png"),
                xlabel="Head",
                ylabel="Layer",
                cmap="coolwarm",
            )
            
            # Advanced 3D Head plotting mapping
            surface_3d_plot(
                mat,
                [str(x) for x in heads],
                [str(x) for x in layers],
                f"{mode.title()} Consistency-Layer Head Delta (3D Profile)",
                os.path.join(PLOTS_DIR, f"{mode}_consistency_head_3d.png")
            )

            h.sort_values("delta_prefix_attn_head", ascending=False).head(12).to_csv(
                os.path.join(CSV_DIR, f"{mode}_top_heads.csv"), index=False
            )

    for mode in ["anchored", "plain"]:
        if not token_df.empty:
            df = token_df[token_df["mode"] == mode]
            if len(df) > 0:
                plt.figure(figsize=(8, 4))
                plt.hist(df["clean_prompt_tokens"], bins=10, alpha=0.7, label="clean")
                plt.hist(df["corrupt_prompt_tokens"], bins=10, alpha=0.7, label="corrupt")
                plt.title(f"{mode.title()} prompt token lengths")
                plt.xlabel("Tokens")
                plt.ylabel("Count")
                plt.legend()
                plt.tight_layout()
                plt.savefig(os.path.join(PLOTS_DIR, f"{mode}_token_length_hist.png"), dpi=180)
                plt.close()

    md = ["# Consistency-head experiment report\n"]
    md.append(f"- Model: `{BASE_MODEL}`")
    md.append(f"- Samples: {N_SAMPLES}")
    md.append(f"- Consistency layers studied: {CONSISTENCY_LAYERS}")
    md.append(f"- Prefix attention window: {PREFIX_WINDOW}")
    md.append(f"- Patch window: {PATCH_WINDOW}")
    md.append("\n## Summary table\n")
    md.append("| Mode | Clean acc | Corrupt acc | Clean logprob | Corrupt logprob | Delta logprob |")
    md.append("|---|---:|---:|---:|---:|---:|")
    for _, r in summary_df.iterrows():
        md.append(
            f"| {r['mode']} | {r['clean_accuracy']:.3f} | {r['corrupt_accuracy']:.3f} | "
            f"{r['clean_gold_logprob_mean']:.3f} | {r['corrupt_gold_logprob_mean']:.3f} | {r['delta_gold_logprob_mean']:.3f} |"
        )

    md.append("\n## Observations\n")
    for mode in ["anchored", "plain"]:
        if not patch_agg.empty:
            p = patch_agg[patch_agg["mode"] == mode]
            best_patch_layer = int(p.sort_values("recovery", ascending=False).iloc[0]["layer"]) if len(p) else -1
            best_patch_recovery = float(p.sort_values("recovery", ascending=False).iloc[0]["recovery"]) if len(p) else float("nan")
        else:
            best_patch_layer, best_patch_recovery = -1, float("nan")

        if not dla_agg.empty:
            d = dla_agg[dla_agg["mode"] == mode]
            best_dla_layer = int(d.sort_values("delta_dla", ascending=False).iloc[0]["layer"]) if len(d) else -1
            best_dla_delta = float(d.sort_values("delta_dla", ascending=False).iloc[0]["delta_dla"]) if len(d) else float("nan")
        else:
            best_dla_layer, best_dla_delta = -1, float("nan")

        if not attn_agg.empty:
            a = attn_agg[attn_agg["mode"] == mode]
            best_attn_layer = int(a.sort_values("delta_prefix_attn", ascending=False).iloc[0]["layer"]) if len(a) else -1
            best_attn_delta = float(a.sort_values("delta_prefix_attn", ascending=False).iloc[0]["delta_prefix_attn"]) if len(a) else float("nan")
        else:
            best_attn_layer, best_attn_delta = -1, float("nan")

        md.append(f"- **{mode}**: strongest patch recovery at layer {best_patch_layer} (recovery {best_patch_recovery:.3f}).")
        md.append(f"- **{mode}**: strongest DLA delta at layer {best_dla_layer} (delta {best_dla_delta:.3f}).")
        md.append(f"- **{mode}**: strongest attention-prefix delta at layer {best_attn_layer} (delta {best_attn_delta:.3f}).")

    md.append("\n## Files saved\n")
    md.append("- `csv/samples.csv`")
    md.append("- `csv/patch_rows.csv`")
    md.append("- `csv/attention_rows.csv`")
    md.append("- `csv/eap_rows.csv`")
    md.append("- `csv/tokenization.csv`")
    md.append("- `csv/dla_rows.csv`")
    md.append("- `plots/*.png`")
    md.append("- `reports/summary.json`, `reports/report.md`")

    with open(os.path.join(REPORTS_DIR, "report.md"), "w") as f:
        f.write("\n".join(md))

    save_json(summary, os.path.join(REPORTS_DIR, "summary.json"))

    return summary_df, patch_agg, eap_agg, attn_agg, dla_agg

# ============================================================
# MAIN
# ============================================================

def main():
    if not HAS_PEFT:
        print("PEFT not installed; the script can still run if the SFT adapter path is already merged.")

    print("Loading model and tokenizer ...")
    load_model()

    print("Loading GSM8K samples ...")
    samples = load_gsm8k_samples(N_SAMPLES)

    raw_rows = []
    patch_rows = []
    attn_rows = []
    eap_rows = []
    token_rows = []
    dla_rows = []

    for i, sample in enumerate(samples):
        print(f"Sample {i+1}/{len(samples)} | {preview_text(sample['question'], 90)}")
        srows, prows, arows, erows, trows, drows = run_sample(sample)
        raw_rows.extend(srows)
        patch_rows.extend(prows)
        attn_rows.extend(arows)
        eap_rows.extend(erows)
        token_rows.extend(trows)
        dla_rows.extend(drows)
        free_memory()

    samples_df = pd.DataFrame(raw_rows)
    patch_df = pd.DataFrame(patch_rows)
    attn_df = pd.DataFrame(attn_rows)
    eap_df = pd.DataFrame(eap_rows)
    token_df = pd.DataFrame(token_rows)
    dla_df = pd.DataFrame(dla_rows)

    summary_df, patch_agg, eap_agg, attn_agg, dla_agg = aggregate_and_save(
        samples_df, patch_df, attn_df, eap_df, token_df, dla_df
    )

    print("\n===== SUMMARY =====")
    if not summary_df.empty:
        print(summary_df.to_string(index=False))
    print("\nSaved to:")
    print(f"- {OUT_DIR}")
    print(f"- plots: {PLOTS_DIR}")
    print(f"- csv: {CSV_DIR}")
    print(f"- reports: {REPORTS_DIR}")

if __name__ == "__main__":
    main()

Loading model and tokenizer ...


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Sample 1/40 | Carol and Jennifer are sisters from Los Angeles who love collecting signatures from celebr...
Sample 2/40 | A team of 4 painters worked on a mansion for 3/8ths of a day every day for 3 weeks. How ma...
Sample 3/40 | It costs $194 per meter to repave a street. Monica's street is 150 meters long. How much m...
Sample 4/40 | Richard lives in an apartment building with 15 floors. Each floor contains 8 units, and 3/...
Sample 5/40 | An ice cream truck is traveling through a neighborhood. Children from various homes have s...
Sample 6/40 | James runs 12 miles a day for 5 days a week.  If he runs 10 miles an hour how many hours d...
Sample 7/40 | Mark was unwell for 3 months, during which he lost 10 pounds per month. If his final weigh...
Sample 8/40 | Craig and his brother take turns spelling out the longest letter words they know and count...
Sample 9/40 | Vincent can buy flowers in packages of 3 for $2.50 or in packages of 2 for $1. How much mo...
Sample 10/40 | Mario needs t

/tmp/ipykernel_23/2522227431.py:882: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  attn_layer_df = attn_df[~attn_df.get("is_head_row", False).fillna(False)] if "is_head_row" in attn_df.columns else attn_df.copy()
/tmp/ipykernel_23/2522227431.py:270: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()
/tmp/ipykernel_23/2522227431.py:270: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()



===== SUMMARY =====
    mode  n  clean_accuracy  corrupt_accuracy  clean_gold_logprob_mean  corrupt_gold_logprob_mean  delta_gold_logprob_mean  delta_target_logit_mean  avg_clean_prompt_len  avg_corrupt_prompt_len
anchored 40           0.125              0.15                -1.449319                  -1.445846                -0.003472                 0.022656                196.45                   196.7
   plain 40           0.150              0.15                -1.804763                  -1.796247                -0.008516                -0.038281                192.45                   192.7

Saved to:
- /kaggle/working/phi3_consistency_head_eap
- plots: /kaggle/working/phi3_consistency_head_eap/plots
- csv: /kaggle/working/phi3_consistency_head_eap/csv
- reports: /kaggle/working/phi3_consistency_head_eap/reports
